# K-Means Clustering
Unsupervised algorithm: partition data into K clusters by repeating
(1) assign each point to nearest centroid, (2) move centroid to the mean of its points,
until centroids stop moving. Minimizes inertia (within-cluster sum of squares).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=7)
plt.scatter(X[:, 0], X[:, 1], s=10); plt.title("Unlabelled data"); plt.show()

## 1. K-Means from scratch (Lloyd's algorithm)

In [ ]:
def my_kmeans(X, k, max_iter=100, seed=0):
    rng = np.random.RandomState(seed)
    centroids = X[rng.choice(len(X), k, replace=False)]
    for i in range(max_iter):
        dist = np.linalg.norm(X[:, None] - centroids[None], axis=2)   # (n, k) distances
        labels = dist.argmin(axis=1)                                   # assignment step
        new = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centroids[j] for j in range(k)])
        if np.allclose(new, centroids):                                # converged
            break
        centroids = new                                                # update step
    inertia = ((X - centroids[labels]) ** 2).sum()
    return labels, centroids, inertia, i + 1

labels, cent, inertia, iters = my_kmeans(X, 4)
print(f"Converged in {iters} iterations, inertia = {inertia:.1f}")
plt.scatter(X[:, 0], X[:, 1], c=labels, s=10, cmap="viridis")
plt.scatter(*cent.T, c="red", marker="X", s=200, edgecolors="k"); plt.title("Scratch K-Means")
plt.show()

## 2. Choosing K: Elbow method and Silhouette score

In [ ]:
ks = range(2, 11)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertias.append(km.inertia_); sils.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertias, "o-"); ax[0].set(xlabel="K", ylabel="Inertia", title="Elbow method")
ax[1].plot(ks, sils, "o-", c="g"); ax[1].set(xlabel="K", ylabel="Silhouette", title="Silhouette score (higher is better)")
plt.show()
print("Best K by silhouette:", list(ks)[int(np.argmax(sils))])

## 3. Initialization: k-means++ vs random

In [ ]:
for init in ["random", "k-means++"]:
    inert = [KMeans(4, init=init, n_init=1, random_state=s).fit(X).inertia_ for s in range(20)]
    print(f"{init:10s} mean inertia={np.mean(inert):.1f}, worst={np.max(inert):.1f}")

## 4. Final model and cluster summary

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
print("Cluster sizes:", np.bincount(km.labels_))
print("Centroids:\n", km.cluster_centers_.round(2))
print("Silhouette:", round(silhouette_score(X, km.labels_), 3))
print("Predict new points:", km.predict([[0, 0], [5, 5]]))

## 5. Limitations: K-Means assumes round, similar-sized clusters; scale your features

In [ ]:
Xm, _ = make_moons(400, noise=0.07, random_state=0)
lab = KMeans(2, n_init=10, random_state=0).fit_predict(Xm)
plt.scatter(Xm[:, 0], Xm[:, 1], c=lab, s=10); plt.title("K-Means fails on non-spherical (moon) clusters")
plt.show()

Xu = np.c_[X[:, 0], X[:, 1] * 1000]            # one feature on a huge scale dominates distance
print("Unscaled silhouette:", round(silhouette_score(Xu, KMeans(4, n_init=10, random_state=0).fit_predict(Xu)), 3))
Xsc = StandardScaler().fit_transform(Xu)
print("Scaled silhouette  :", round(silhouette_score(Xsc, KMeans(4, n_init=10, random_state=0).fit_predict(Xsc)), 3))